In [ ]:
import argparse
import logging
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from concurrent.futures.process import BrokenProcessPool
from functools import partial
from pathlib import Path

import pandas as pd
from rdkit.Chem import QED, Crippen
from rdkit.Chem.rdchem import Mol
from rdkit.Chem.rdMolDescriptors import CalcNumRotatableBonds
from rdkit.Chem.rdmolfiles import MolFromMolBlock
from rdkit.Chem.rdmolops import AddHs, CombineMols, GetMolFrags, RemoveAllHs, RemoveHs, SanitizeMol
from rdkit.rdBase import LogToPythonLogger
from rich.logging import RichHandler
from rich.progress import BarColumn, MofNCompleteColumn, Progress, TaskProgressColumn, TextColumn, TimeRemainingColumn, track


In [ ]:
file = "/homes/buttensc/Projects/semla-flow/predictions/unconditional/geoldm/geoldm_100000_predictions.sdf"
with open(file) as filehandle:
    blocks = filehandle.read().rstrip().rstrip("\n").rstrip("\n").rstrip("$$$$").split("$$$$\n")
blocks = blocks[:100]

In [ ]:
def count_radicals(mol: Mol) -> Mol:
    """Count the number of radicals in a molecule."""

    return sum(atom.GetNumRadicalElectrons() for atom in mol.GetAtoms())


def hydrate_radicals(mol: Mol) -> Mol:
    """Hydrate radicals in a molecule."""

    SanitizeMol(mol)
    for atom in mol.GetAtoms():
        num_atom_radicals = atom.GetNumRadicalElectrons()
        if num_atom_radicals:
            atom.SetNumExplicitHs(atom.GetNumExplicitHs() + num_atom_radicals)
            atom.SetNumRadicalElectrons(0)
    SanitizeMol(mol)
    return mol

In [ ]:
results = []
for block in blocks:
    result = {}
    try:
        mol = MolFromMolBlock(block, sanitize=False, removeHs=False, strictParsing=False)
        mol.RemoveAllConformers()
        SanitizeMol(mol, catchErrors=False)
        result["sanitizes"] = 1
        result["all_hydrogens"] = int((AddHs(mol).GetNumAtoms() - mol.GetNumAtoms()) == 0)
        result["no_radicals"] = int(count_radicals(mol) == 0)
        result["connected"] = int(len(GetMolFrags(mol, asMols=False, sanitizeFrags=False)) == 1)
    except:
        result["sanitizes"] = 0
        result["all_hydrogens"] = pd.NA
        result["no_radicals"] = pd.NA
        result["connected"] = pd.NA
    results.append(result)

In [ ]:
df = pd.DataFrame(results)
df.mean()


In [ ]:
df.head()

In [ ]:
block = blocks[0]
mol = MolFromMolBlock(block, sanitize=False, removeHs=False, strictParsing=False)
mol.RemoveAllConformers()
SanitizeMol(mol, catchErrors=False)
mol

The molecule is missing some explicit Hydrogens.

In [ ]:
mol = MolFromMolBlock(block, sanitize=True, removeHs=True, strictParsing=True)
mol.RemoveAllConformers()
mol

The molecule is still missing some explicit Hydrogens.

In [ ]:
mol = MolFromMolBlock(block, sanitize=False, removeHs=False, strictParsing=False)
mol.RemoveAllConformers()
mol = hydrate_radicals(mol)
mol

In [ ]:
count_radicals(mol)